# Ingest: Medicare Beneficiary Summary

Healthcare demo data. Downloads a sample of CMS's DE-SynPUF (De-identified
Synthetic Public Use File) — realistic-shaped, fully synthetic Medicare
beneficiary data safe for public demos — and persists it as a managed
Spark table.

Companion to `medicare_inpatient_claims_ingest.ipynb` — both feed
`data-transformation/readmission_risk_summary.ipynb`.

Confirmed live (2026-08-07): CMS's server blocks requests with no
`User-Agent` header (403) — a normal browser/tool UA works fine.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "medicare_claims"
TABLE_NAME = "beneficiary_summary"
SOURCE_URL = (
    "https://www.cms.gov/research-statistics-data-and-systems/downloadable-public-use-files/"
    "synpufs/downloads/de1_0_2008_beneficiary_summary_file_sample_1.zip"
)
LOCAL_ZIP = "/tmp/de1_0_2008_beneficiary_summary_file_sample_1.zip"
LOCAL_DIR = "/tmp/medicare_beneficiary"

## Download source file

A `User-Agent` header is required — see note above.

In [ ]:
import urllib.request

request = urllib.request.Request(SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response, open(LOCAL_ZIP, "wb") as out_file:
    out_file.write(response.read())

## Extract CSV

In [ ]:
import glob
import zipfile

with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(LOCAL_DIR)

csv_files = glob.glob(f"{LOCAL_DIR}/*.csv")
csv_files

## Load dataset

Chronic-condition flag columns (`SP_*`) use CMS's coding: `1` = has the
condition, `2` = does not.

In [ ]:
df = spark.read.csv(csv_files, header=True, inferSchema=True)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Synthetic Medicare claims data (CMS DE-SynPUF) used for healthcare demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_beneficiaries FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)